# Creational Design Patterns
### Factory Method | Abstract Factory | Builder | Prototype | Singleton

> **One coherent system throughout:** ShopFlow -- a 500k-user e-commerce platform.
> Every pattern is anchored to a **real production incident**.
>
> Format per pattern: **Mental Model -> Real Scenario -> BEFORE (broken) -> AFTER (fixed) -> Frameworks**

*Run each cell with **Shift + Enter***

## Setup

In [ ]:
from __future__ import annotations
import copy, threading, time
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any

---
## 1 · Singleton

### Mental Model -- 'The White House Hotline'

```
WHAT   One instance shared by every caller, forever.
WHY    Shared state (config, pools, caches) must be consistent.
HOW    The class controls its own construction and caches the first result.
WHEN   Config objects | DB-connection pools | loggers | registries
       WARNING: NOT for services that need unit-test isolation --
       use dependency injection instead.
```

```
First call:        Singleton.__new__ -> creates instance -> stores in _inst
All later calls:   Singleton.__new__ -> returns _inst    (no new object)
```

**Anti-pattern alert:** Singleton is the most-abused pattern. Use it only
when the identity of the instance genuinely matters (same connection pool,
same in-memory cache). For stateless services, prefer module-level
singletons or DI containers.

### Real-World Scenario -- ShopFlow Config Manager

**Incident:** Three microservices each called `Config()` at import time.
After a hot-reload the `payment` service picked up a new Stripe key
but the old instance still lived in `inventory` -- payment calls started
failing because the two services disagreed on the API key.

**Root cause:** `Config` was not a singleton -- every module created its own copy.
**Fix:** Thread-safe singleton -- all code shares the exact same object.

In [ ]:
# BEFORE -- every import creates a new Config object

class Config_BAD:
    def __init__(self) -> None:
        self.stripe_key = 'sk_live_ABC'
        self.db_url     = 'postgresql://prod'

cfg1 = Config_BAD()
cfg2 = Config_BAD()
cfg1.stripe_key = 'sk_live_NEW'      # hot-reload updated this one
print('Same object?', cfg1 is cfg2)  # False -- inventory has stale key!
print('cfg2 key:', cfg2.stripe_key)  # sk_live_ABC  <- STALE

In [ ]:
# AFTER -- thread-safe Singleton via __new__

class Config:
    _instance: 'Config | None' = None
    _lock = threading.Lock()

    def __new__(cls) -> 'Config':
        if cls._instance is None:
            with cls._lock:                         # double-checked locking
                if cls._instance is None:
                    inst = super().__new__(cls)
                    inst.stripe_key = 'sk_live_ABC'
                    inst.db_url     = 'postgresql://prod'
                    cls._instance   = inst
        return cls._instance

    def reload(self, key: str) -> None:
        self.stripe_key = key           # mutates THE shared instance


cfg_a = Config()
cfg_b = Config()
cfg_a.reload('sk_live_NEW')

assert cfg_a is cfg_b
assert cfg_b.stripe_key == 'sk_live_NEW'   # hot-reload visible everywhere!
print('Same object:', id(cfg_a) == id(cfg_b))

### Where This Is Seen in Real Frameworks

| Framework | Singleton usage |
|-----------|----------------|
| **Django** | `django.conf.settings` -- one settings object, imported everywhere |
| **SQLAlchemy** | `create_engine()` -- one engine per app; never call it per request |
| **Python logging** | `logging.getLogger('name')` -- same logger by name, always |
| **FastAPI** | `lifespan` startup hook -- create one DB pool, store on `app.state` |
| **Celery** | `app = Celery(...)` -- module-level singleton used by all tasks |

> **Pythonic tip:** A module is itself a singleton. `import config` always
> returns the same module object. Prefer this over a class with `__new__`.

---
## 2 · Factory Method

### Mental Model -- 'The Restaurant Order'

```
WHAT   A single function decides WHICH concrete class to build;
       callers never see ConcreteClass() directly.
WHY    Adding a new type requires zero changes at call sites.
HOW    A dict registry maps a key -> class; factory instantiates.
WHEN   Multi-format exporters | storage drivers | notification channels.
```

```
Client                 Factory               Product
------                 -------               -------
give me 'pdf'  -----> make_report('pdf') -> PdfReport()
give me 'csv'  -----> make_report('csv') -> CsvReport()
give me 'json' -----> make_report('json')-> JsonReport()

Client never imports PdfReport, CsvReport, JsonReport directly.
```

**Key insight:** The factory is the ONLY place that knows concrete classes.
Open for extension (add new type -> one registry entry),
closed for modification (call sites unchanged) -- this IS the OCP in action.

### Real-World Scenario -- ShopFlow Report Generator

**Incident:** The ops team needed to switch from PDF to Excel reports mid-sprint.
The old code had `if fmt == 'pdf': PdfReport()` scattered across 12 endpoints.
Changing the format required touching every endpoint.

**Fix:** A single `make_report(fmt)` factory. Change one registry entry.

In [ ]:
# BEFORE -- format logic scattered across every caller

def export_orders_BAD(orders: list, fmt: str) -> bytes:
    if fmt == 'pdf':
        return b'<pdf bytes>'     # 40 lines of PDF code here
    elif fmt == 'csv':
        return b'<csv bytes>'
    elif fmt == 'xlsx':
        return b'<xlsx bytes>'
    # Adding 'parquet' means editing THIS function AND all other
    # callers that have the same if/elif block.
    raise ValueError(f'Unknown format: {fmt}')

In [ ]:
# AFTER -- Factory Method with a registry
import json, io, csv

class ReportBase(ABC):
    @abstractmethod
    def render(self, data: list[dict]) -> bytes: ...

class PdfReport(ReportBase):
    def render(self, data):
        rows = '\n'.join(str(r) for r in data)
        return f'[PDF]\n{rows}'.encode()

class CsvReport(ReportBase):
    def render(self, data):
        buf = io.StringIO()
        if data:
            w = csv.DictWriter(buf, fieldnames=data[0])
            w.writeheader(); w.writerows(data)
        return buf.getvalue().encode()

class JsonReport(ReportBase):
    def render(self, data):
        return json.dumps(data, indent=2).encode()

# The registry is the ONLY place that knows about concrete classes
_REGISTRY: dict[str, type[ReportBase]] = {
    'pdf':  PdfReport,
    'csv':  CsvReport,
    'json': JsonReport,
}

def make_report(fmt: str) -> ReportBase:
    cls = _REGISTRY.get(fmt)
    if cls is None:
        raise ValueError(f'Unknown format {fmt!r}. Available: {list(_REGISTRY)}')
    return cls()

data = [{'id': 1, 'item': 'Widget', 'qty': 10}]
for fmt in ('pdf', 'csv', 'json'):
    output = make_report(fmt).render(data)
    print(f'[{fmt}] {output[:60]}')

### Where This Is Seen in Real Frameworks

| Framework | Factory Method usage |
|-----------|---------------------|
| **Django** | `Model.objects.create()` -- manager is the factory |
| **SQLAlchemy** | `sessionmaker()` -- returns a Session factory callable |
| **FastAPI** | `Depends(get_db)` -- dependency injection is a factory per-request |
| **Pydantic v2** | `model_validator(mode='before')` -- factory hook for coercion |
| **Celery** | `@app.task` registers a factory for task classes |

---
## 3 · Abstract Factory

### Mental Model -- 'The IKEA Room Package'

```
WHAT   A factory that creates FAMILIES of related objects.
       Switch the whole family at once, not individual pieces.
WHY    AWS S3 + SQS must go together; never mix AWS storage with GCP queue.
HOW    An interface declares factory methods; concrete factories implement it.
WHEN   Multi-cloud adapters | UI themes | test vs production backends
```

```
            CloudFactory (ABC)
              storage() -> StorageBase
              queue()   -> QueueBase
             /                    \
        AwsFactory            GcpFactory
       storage()->S3        storage()->GCS
       queue() ->SQS        queue() ->PubSub
```

### Real-World Scenario -- ShopFlow Multi-Cloud Failover

**Incident:** ShopFlow launched on AWS. After a 4-hour outage they needed GCP
failover. 47 files imported `boto3` directly -- the migration took 3 weeks.

**Fix:** All cloud interactions go through `CloudFactory`.
Switching cloud = change one env-var.

In [ ]:
# BEFORE -- cloud SDK calls scattered everywhere
import os

def upload_image_BAD(key: str, data: bytes) -> str:
    provider = os.getenv('CLOUD', 'aws')
    if provider == 'aws':
        # boto3.client('s3').put_object(...)  <- real call
        return f's3://shopflow-prod/{key}'
    elif provider == 'gcp':
        # gcs.Client().bucket(...).blob(...).upload_from_string(data)
        return f'gs://shopflow-prod/{key}'
    raise RuntimeError(f'Unknown provider {provider}')
# Same if/elif repeated in 47 functions.

In [ ]:
# AFTER -- Abstract Factory: swap the whole cloud family at once
import os

class StorageBase(ABC):
    @abstractmethod
    def upload(self, key: str, data: bytes) -> str: ...

class QueueBase(ABC):
    @abstractmethod
    def publish(self, topic: str, msg: str) -> None: ...

class S3Storage(StorageBase):
    def upload(self, key, data): return f's3://shopflow-prod/{key}'

class SqsQueue(QueueBase):
    def publish(self, topic, msg): print(f'  [SQS] -> {topic}: {msg}')

class GcsStorage(StorageBase):
    def upload(self, key, data): return f'gs://shopflow-prod/{key}'

class PubSubQueue(QueueBase):
    def publish(self, topic, msg): print(f'  [PubSub] -> {topic}: {msg}')

class CloudFactory(ABC):
    @abstractmethod
    def storage(self) -> StorageBase: ...
    @abstractmethod
    def queue(self) -> QueueBase: ...

class AwsFactory(CloudFactory):
    def storage(self): return S3Storage()
    def queue(self):   return SqsQueue()

class GcpFactory(CloudFactory):
    def storage(self): return GcsStorage()
    def queue(self):   return PubSubQueue()

# Application code never mentions AWS or GCP directly
def upload_and_notify(factory: CloudFactory, key: str, data: bytes) -> str:
    url = factory.storage().upload(key, data)
    factory.queue().publish('image-uploaded', url)
    return url

factory = AwsFactory() if os.getenv('CLOUD', 'aws') == 'aws' else GcpFactory()
print('AWS:', upload_and_notify(factory, 'products/shoes.jpg', b'<img>'))
print('GCP:', upload_and_notify(GcpFactory(), 'products/shoes.jpg', b'<img>'))

### Where This Is Seen in Real Frameworks

| Framework | Abstract Factory usage |
|-----------|------------------------|
| **SQLAlchemy** | Database dialects -- `create_engine('postgresql://')` vs `'sqlite://'` swaps the whole SQL-generation family |
| **Django** | `DATABASES` backend -- swap `django.db.backends.postgresql` -> `sqlite3` |
| **Pytest** | `conftest.py` fixtures -- factory functions create mock vs real backends |
| **Boto3 / Moto** | `boto3.resource('s3')` returns local mock in tests; same interface, different factory |
| **httpx** | `AsyncClient` vs `Client` -- swap transport factory for mock transport in tests |

---
## 4 · Builder

### Mental Model -- 'The Subway Order'

```
WHAT   Separate construction of a complex object from its representation.
       Build it step by step; each step is optional and order-safe.
WHY    Constructors with 10+ parameters are unreadable and error-prone.
       'What does True mean in position 7?' is a real bug.
HOW    Builder object accumulates config via chained methods.
       .build() validates and returns the product.
WHEN   HTTP request builders | SQL query builders | email builders
```

```
EmailBuilder()
  .to('alice@shopflow.com')      # recipient -- method makes intent clear
  .subject('Order shipped!')     # required
  .bcc('ops@shopflow.com')       # clearly named -- impossible to swap with reply_to
  .reply_to('support@...')       # clearly named
  .priority('high')              # optional
  .build()                       # validates required fields
```

### Real-World Scenario -- ShopFlow Email Service

**Incident:** `send_email(to, subject, body, cc, bcc, reply_to, attachments,
priority, template_id, unsubscribe_url, tracking_enabled)` -- 11 positional args.
A dev swapped `bcc` and `reply_to` and ShopFlow sent 80k marketing emails
to real addresses that were supposed to be BCC'd.

**Fix:** Fluent `EmailBuilder` -- named methods make intent explicit,
impossible to mix up positions, `.build()` validates required fields.

In [ ]:
# BEFORE -- telescoping constructor (11 positional args)

def send_email_BAD(to, subject, body, cc=None, bcc=None,
                   reply_to=None, attachments=None,
                   priority='normal', tracking=True):
    pass

# Called like this -- good luck knowing which arg is which:
send_email_BAD(
    'alice@shopflow.com',
    'Your order is shipped',
    'Body here...',
    None,                    # cc
    'ops@shopflow.com',      # <- INTENDED as BCC
    'support@shopflow.com',  # <- INTENDED as reply_to
    None, 'high', True,      # swapped? hard to tell
)

In [ ]:
# AFTER -- Fluent Builder

@dataclass
class Email:
    to:          str
    subject:     str
    body:        str
    cc:          list[str] = field(default_factory=list)
    bcc:         list[str] = field(default_factory=list)
    reply_to:    str | None = None
    attachments: list[str] = field(default_factory=list)
    priority:    str = 'normal'
    tracking:    bool = True


class EmailBuilder:
    def __init__(self) -> None:
        self._to, self._subject, self._body = None, None, None
        self._cc, self._bcc = [], []
        self._reply_to, self._attachments = None, []
        self._priority, self._tracking = 'normal', True

    # Each method returns self -> fluent chaining
    def to(self, addr: str)       -> 'EmailBuilder': self._to = addr;            return self
    def subject(self, s: str)     -> 'EmailBuilder': self._subject = s;          return self
    def body(self, b: str)        -> 'EmailBuilder': self._body = b;             return self
    def cc(self, addr: str)       -> 'EmailBuilder': self._cc.append(addr);      return self
    def bcc(self, addr: str)      -> 'EmailBuilder': self._bcc.append(addr);     return self
    def reply_to(self, a: str)    -> 'EmailBuilder': self._reply_to = a;         return self
    def attach(self, p: str)      -> 'EmailBuilder': self._attachments.append(p);return self
    def priority(self, p: str)    -> 'EmailBuilder': self._priority = p;         return self
    def no_tracking(self)         -> 'EmailBuilder': self._tracking = False;     return self

    def build(self) -> Email:
        if not self._to:      raise ValueError('Email must have a recipient')
        if not self._subject: raise ValueError('Email must have a subject')
        if not self._body:    raise ValueError('Email must have a body')
        return Email(to=self._to, subject=self._subject, body=self._body,
                     cc=self._cc, bcc=self._bcc, reply_to=self._reply_to,
                     attachments=self._attachments, priority=self._priority,
                     tracking=self._tracking)


email = (
    EmailBuilder()
    .to('alice@shopflow.com')
    .subject('Your order #1234 is shipped!')
    .body('Track it at shopflow.com/track/1234')
    .bcc('ops@shopflow.com')            # clearly BCC
    .reply_to('support@shopflow.com')   # clearly reply_to
    .attach('invoice.pdf')
    .priority('high')
    .build()
)
print(email)

### Where This Is Seen in Real Frameworks

| Framework | Builder usage |
|-----------|______________|
| **SQLAlchemy** | `session.query(User).filter(...).order_by(...).limit(10).all()` |
| **Django ORM** | `User.objects.filter(active=True).exclude(role='admin').select_related()` |
| **httpx** | `client.get(url, headers={}, params={}, timeout=5)` |
| **FastAPI** | `APIRouter(prefix='/api/v1', tags=['orders'], dependencies=[...])` |
| **Pydantic** | `model_config = ConfigDict(strict=True, frozen=True)` |

---
## 5 · Prototype

### Mental Model -- 'The Cookie Cutter'

```
WHAT   Create new objects by CLONING a pre-configured 'template' instance.
WHY    Initialization is expensive; many similar objects differ only in details.
HOW    copy.copy() (shallow) or copy.deepcopy() (deep) the prototype.
WHEN   Game entities | report templates | Kubernetes pod specs |
       notification templates
```

```
     Prototype (fully configured, shared SMTP + tracking + unsubscribe)
          |          clone()
   +------+------+
   v      v      v
 notif_A notif_B notif_C
 (each customizes only to/subject/template -- shared config is correct everywhere)
```

**Shallow vs Deep copy:**
- `copy.copy()` -- copies the object; nested objects are still shared
- `copy.deepcopy()` -- copies everything recursively (safe but slower)

### Real-World Scenario -- ShopFlow Notification Templates

**Incident:** ShopFlow sends 5 types of transactional emails. Each required
the same corporate header, footer, tracking config, and SMTP settings.
Engineers copy-pasted setup code and forgot to update the unsubscribe URL
in the refund email -- violating CAN-SPAM.

**Fix:** One `base_notification` prototype. Each email type clones and customizes it.

In [ ]:
# BEFORE -- copy-paste setup for every notification type

def build_order_placed_email_BAD(order_id: str, email: str) -> dict:
    return {
        'from':         'no-reply@shopflow.com',
        'reply_to':     'support@shopflow.com',
        'smtp_host':    'smtp.sendgrid.net',
        'smtp_port':    587,
        'track_opens':  True,
        'track_clicks': True,
        'unsubscribe':  'https://shopflow.com/unsub',  # forgotten in refund email
        'to':           email,
        'subject':      f'Order #{order_id} placed!',
        'template':     'order_placed',
    }
# Duplicate this 4 more times. One typo in unsubscribe = CAN-SPAM violation.

In [ ]:
# AFTER -- Prototype Pattern: clone and customize

@dataclass
class NotificationConfig:
    sender:         str = 'no-reply@shopflow.com'
    reply_to:       str = 'support@shopflow.com'
    smtp_host:      str = 'smtp.sendgrid.net'
    smtp_port:      int = 587
    track_opens:    bool = True
    track_clicks:   bool = True
    unsubscribe:    str = 'https://shopflow.com/unsub'
    to:             str = ''
    subject:        str = ''
    template:       str = ''
    extra_data:     dict = field(default_factory=dict)

    def clone(self) -> 'NotificationConfig':
        return copy.deepcopy(self)   # deep copy: nested dicts are independent


# Build the prototype ONCE with all shared config:
_base = NotificationConfig()

def build_order_placed(order_id: str, recipient: str) -> NotificationConfig:
    n = _base.clone()
    n.to, n.subject, n.template = recipient, f'Order #{order_id} placed!', 'order_placed'
    n.extra_data = {'order_id': order_id}
    return n

def build_refund(order_id: str, amount: float, recipient: str) -> NotificationConfig:
    n = _base.clone()
    n.to, n.subject, n.template = recipient, f'Refund of ${amount:.2f} processed', 'refund'
    n.extra_data = {'order_id': order_id, 'amount': amount}
    return n


placed = build_order_placed('A-1234', 'alice@shopflow.com')
refund = build_refund('A-1234', 29.99, 'alice@shopflow.com')

print('Placed:', placed.subject)
print('Refund:', refund.subject)
print('Same unsubscribe URL:', placed.unsubscribe == refund.unsubscribe)  # True
print('Independent extra_data:', placed.extra_data is not refund.extra_data)  # True

### Where This Is Seen in Real Frameworks

| Framework | Prototype usage |
|-----------|----------------|
| **Python stdlib** | `copy.deepcopy()` -- the standard prototype mechanism |
| **Django** | `instance.pk = None; instance.save()` -- clone a model row |
| **Pydantic** | `model.model_copy(update={'field': new_val})` -- immutable clone with overrides |
| **Kubernetes** | Pod templates -- one spec cloned for every replica |
| **Celery** | `task.s()` (signature) -- a cloneable, partially-applied task |

> **Pydantic `model_copy`** is the most common Python prototype in production:
> ```python
> updated = original_order.model_copy(update={'status': 'shipped'})
> ```